Importing libraries

In [18]:
import requests
import yaml
import pandas as pd
import matplotlib.pyplot as plt
from datetime import date

We start by code to fetch the data from [website] using code curtousy of https://frost.met.no/python_example.html

In [6]:
# Insert your own client ID here

with open("key.yaml", "r") as f:
    config = yaml.safe_load(f)

client_id = config.get("client_ID")

endpoint = 'https://frost.met.no/observations/v0.jsonld'
parameters = {
'sources': 'SN17850',
'elements': 'mean(air_temperature P1D)',
'referencetime': '2025-01-01/2025-12-31',
}

# Issue an HTTP GET request
r = requests.get(endpoint, parameters, auth=(client_id, ''))
# Extract JSON data
json = r.json()

# verifies if data is retrived
if r.status_code == 200:
    data = json['data']
    print('Data retrieved from frost.met.no!')
else:
    print('Error! Returned status code %s' % r.status_code)
    print('Message: %s' % json['error']['message'])
    print('Reason: %s' % json['error']['reason'])


# This will return a Dataframe with all of the observations in a table format
df = pd.DataFrame()
for i in range(len(data)):
    row = pd.DataFrame(data[i]['observations'])
    row['referenceTime'] = data[i]['referenceTime']
    row['sourceId'] = data[i]['sourceId']
    df = df.append(row)

df = df.reset_index()

Data retrieved from frost.met.no!


AttributeError: 'DataFrame' object has no attribute 'append'

Task 4

In [ ]:


def count_heatwaves(station_id: str, station_name: str, start_and_end_date: str) -> int:
    """Collects data from frost.met.no and count the number of heatwaves in a year.
    Generates a summary yml-file.
    
    Args:
        station_id: ID of which wather station to collect data from.
        station_name: Name of the wather station.
        start_and_end_date: Start and end date to collect data from (YYYY-MM-DD/YYYY-MM-DD).
    Returns:
        count_heatwaves: Number of heatwaves in the given time period.
    """

    # Collecting max temperature data from frost.net
    parameters_max = {
    'sources': station_id,
    'elements': 'max(air_temperature P1D)',
    'referencetime': start_and_end_date,
    }

    r_max = requests.get(endpoint, parameters_max, auth=(client_id, ''))

    json_max = r_max.json()

    if r_max.status_code == 200:
        data_max = json_max['data']
        print('Data retrieved from frost.met.no!')
    else:
        print('Error! Returned status code %s' % r.status_code)
        print('Message: %s' % json['error']['message'])
        print('Reason: %s' % json['error']['reason'])

    # Extract dates and max temperatures from list with data
    dates = []
    max_temperatures = []
    
    for day in data_max:
        date = day['referenceTime']

        for observation in day['observations']:
            if observation['timeOffset'] == 'PT0H':
                max_temperature = observation['value']

                dates.append(date[:10])
                max_temperatures.append(max_temperature)

                break

    # Find all heatwaves and add the affected dates to a list
    heatwaves = []
    start_date = None
    end_date = None
    consecutive_days = 0
    count_heatwaves = 0

    for current_date, temperature in zip(dates, max_temperatures):
        if temperature >= 27:
            if consecutive_days == 0:
                start_date = current_date

            consecutive_days += 1

        else:
            if consecutive_days >= 5:
                heatwaves.append(f'{start_date} - {end_date}')
                count_heatwaves += 1

            consecutive_days = 0
            start_date = None

        end_date = current_date

    # Make a summary, formatted as the example file
    summary = {
        'dates': heatwaves,
        'station_ID': station_id,
        'station_name': station_name,
        'year': start_and_end_date.split('/')[0].split('-')[0],
    }

    # Make a yml-file with the summary
    with open('heatwave_summary.yml', 'w') as file:
        yaml.dump(summary, file)

    return count_heatwaves


count_heatwaves('SN17850', 'ÅS', '2025-01-01/2025-12-31')


Data retrieved from frost.met.no!


1

The function count_heatwaves counted one heatwave in Ås in 2025. This makes sense because we remember that it was one heatwave in july 2025 which affected Norway.

Task 5

In [ ]:


def moving_average_temperature(mean_temperatures: list) -> list:
    """Calculate the 7-day moving average for temperature."""
    
    average_temp = []

    for i in range(len(mean_temperatures) - 6):
        seven_day_interval = mean_temperatures[i:i + 7]
        average = sum(seven_day_interval) / 7
        average_temp.append(average)

    return average_temp

